# Phase 4 — sensitivity on the δ-ball, not at the clean point

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/phase4_shifted_spectrum.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phases 1–3 measured $C = \mathbb{E}[g g^T]$, $g = \nabla_x \ell$, at clean images, and neither $\lambda_1(C)$ nor
$\operatorname{tr} C$ predicted universal vulnerability (Phase 3: median within-setting Spearman ρ 0.46 and 0.43).

**Why that may be.** To second order, $\ell(x+v) - \ell(x) \approx g^T v + \tfrac12 v^T H v$. A clean-point spectrum
sees only $g$. A gradient penalty shrinks $g$ at the clean point and can leave the curvature, while universal
adversarial training fits the model at $x + v$ and so controls the whole neighbourhood. This matches the Phase 2
counterexample: UAT and the trace penalty had the same $\lambda_1$, but the trace model was fooled twice as often.

**Hypothesis.** Universal vulnerability is set by the spectrum of the gradient covariance after moving along the
active direction:

$$C_\delta = \mathbb{E}\big[g(x \pm \delta u_1)\, g(x \pm \delta u_1)^T\big],$$

with $u_1$ the top eigenvector of the clean $C$ and a random sign per image. No attack is run to compute it.
**Primary measure:** $\lambda_1(C_\delta)$ at $\delta = 4$, the attack budget used throughout.
Secondary measures, reported but not used for the decision: $\operatorname{tr} C_\delta$, the same at $\delta = 2$,
and $C$ after a random shift of norm $\delta$.

**Rules, written before the run.**
- The decision uses the 5 settings where the base model learned. DermaMNIST with VGG-19 is excluded (balanced accuracy 27%)
  and is still reported.
- **H1:** the median within-setting Spearman ρ between $\lambda_1(C_\delta)$ and AS fooling (6 models per setting) is ≥ 0.7.
- **H2:** that ρ is higher than the clean-point $\lambda_1$ ρ in at least 4 of the 5 settings.
- **H3** (reported, not required): in at least 4 of the 5 settings, $\lambda_1(C_\delta)$ orders UAT and trace β = 0.1
  the same way as their AS fooling.
- The hypothesis is **supported** if H1 and H2 both hold, and **rejected** otherwise.

**Run.** This only measures the 36 Phase 3 models saved in Google Drive; nothing is trained. About 1–2 hours on a T4.
Run the first cell, then the second (press **Allow**), then *Run all*. Results are saved in
`active_subspace_runs/phase4`; after a disconnect *Run all* continues.

In [ ]:
%cd /content
!rm -rf active-subspace-bloodmnist && git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git log --oneline -1
!pip install -q medmnist

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from asnet.data import CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model
from asnet.utils import get_device
from asnet.spectrum import input_spectrum, shifted_spectrum

ROOT = "/content/drive/MyDrive/active_subspace_runs"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError("Google Drive is NOT connected. Run this cell again and press Allow.")
else:
    ROOT = "runs"
P3 = os.path.join(ROOT, "phase3")
OUT = os.path.join(ROOT, "phase4")
os.makedirs(OUT, exist_ok=True)
CSV = os.path.join(OUT, "phase4_spectra.csv")
if not os.path.exists(os.path.join(P3, "phase3_defenses.csv")):
    raise RuntimeError("Phase 3 results not found in " + P3 + ": this notebook needs the Phase 3 models.")
device = get_device()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("reading Phase 3 from", P3, "| saving to", OUT, "|", device)

## Settings (the same as Phase 3)

In [ ]:
SIZE, MAX_TRAIN, N_SPECTRUM, DELTA = 64, 12000, 2000, 4
EXCLUDED = [("dermamnist", "vgg19")]     # base model did not learn; decided before this run
p3 = pd.read_csv(os.path.join(P3, "phase3_defenses.csv"))
print(len(p3), "Phase 3 models")

def ckpt(*parts):
    return os.path.join(P3, "models", "_".join(str(x) for x in parts).replace(" ", "").replace("=", "") + ".pt")

_spec = {}
def spec_loader(name):
    if name not in _spec:
        train, _, _ = get_datasets(SIZE, name=name)
        if len(train) > MAX_TRAIN:
            train = subset(train, MAX_TRAIN, seed=7)
        _spec[name] = make_loader(subset(train, N_SPECTRUM, seed=10), 100)
    return _spec[name]

## Measure every model

Each row is saved as soon as it is ready.

In [ ]:
rows = pd.read_csv(CSV).to_dict("records") if os.path.exists(CSV) else []
done = {(r["dataset"], r["arch"], r["model"]) for r in rows}

for _, r in p3.iterrows():
    key = (r["dataset"], r["arch"], r["model"])
    if key in done:
        continue
    path = ckpt(r["dataset"], r["arch"], "base" if r["model"] == "none (base)" else r["model"], 0)
    m = build_model(r["arch"], CLASSES[r["dataset"]], SIZE)
    m.load_state_dict(torch.load(path, map_location="cpu")["state_dict"])
    m.to(device).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    L = spec_loader(r["dataset"])
    clean = input_spectrum(m, L, device=device, return_top_vector=True)
    a4 = shifted_spectrum(m, L, DELTA, "active", u1=clean["u1"], device=device)
    a2 = shifted_spectrum(m, L, DELTA / 2, "active", u1=clean["u1"], device=device)
    r4 = shifted_spectrum(m, L, DELTA, "random", device=device)
    row = {"dataset": r["dataset"], "arch": r["arch"], "model": r["model"],
           "clean acc": r["clean acc"], "AS fooling": r["AS fooling"], "UAP fooling": r["UAP fooling"],
           "SGD-UAP fooling": r["SGD-UAP fooling"],
           "lambda1 clean": clean["lambda1"], "trace clean": clean["trace"],
           "lambda1 active d4": a4["lambda1"], "trace active d4": a4["trace"],
           "lambda1 active d2": a2["lambda1"], "trace active d2": a2["trace"],
           "lambda1 random d4": r4["lambda1"], "trace random d4": r4["trace"]}
    rows.append(row); pd.DataFrame(rows).to_csv(CSV, index=False)
    print(key, {k: round(v, 4) for k, v in row.items() if isinstance(v, float)})
    del m; torch.cuda.empty_cache()

res = pd.DataFrame(rows)
res.round(4)

## The decision

In [ ]:
measures = ["lambda1 clean", "trace clean", "lambda1 active d4", "trace active d4",
            "lambda1 active d2", "trace active d2", "lambda1 random d4", "trace random d4"]
corr = []
for (ds, a), grp in res.groupby(["dataset", "arch"]):
    row = {"dataset": ds, "arch": a, "n": len(grp), "excluded": (ds, a) in EXCLUDED}
    for meas in measures:
        row[meas] = spearmanr(grp[meas], grp["AS fooling"])[0]
    g = grp.set_index("model")
    if {"UAT", "trace beta=0.1"} <= set(g.index):
        row["H3 order ok"] = bool(np.sign(g.loc["UAT", "lambda1 active d4"] - g.loc["trace beta=0.1", "lambda1 active d4"])
                                  == np.sign(g.loc["UAT", "AS fooling"] - g.loc["trace beta=0.1", "AS fooling"]))
    corr.append(row)
corr = pd.DataFrame(corr)
corr.to_csv(os.path.join(OUT, "phase4_correlations.csv"), index=False)
display(corr.round(3))

main = corr[~corr.excluded]
med = main["lambda1 active d4"].median()
better = int((main["lambda1 active d4"] > main["lambda1 clean"]).sum())
h3 = int(main["H3 order ok"].sum())
lines = [
    f"H1: median rho(lambda1 at x +- delta u1, AS fooling) = {med:.3f} over {len(main)} settings (need >= 0.7) -> {'MET' if med >= 0.7 else 'NOT MET'}",
    f"H2: better than clean lambda1 in {better}/{len(main)} settings (need >= 4) -> {'MET' if better >= 4 else 'NOT MET'}",
    f"H3 (reported): orders UAT vs trace beta=0.1 correctly in {h3}/{len(main)} settings",
    "HYPOTHESIS " + ("SUPPORTED" if med >= 0.7 and better >= 4 else "REJECTED"),
    "medians over the main settings: " + ", ".join(f"{m} {main[m].median():.2f}" for m in measures),
]
print("\n".join(lines))
open(os.path.join(OUT, "phase4_decision.txt"), "w").write("\n".join(lines))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
colors = plt.cm.tab10(np.arange(6))
for ax, meas, lab in [(axes[0], "lambda1 clean", r"$\lambda_1(C)$ at clean images"),
                      (axes[1], "lambda1 active d4", r"$\lambda_1(C_\delta)$ at $x \pm \delta u_1$")]:
    for c, ((ds, a), grp) in zip(colors, res.groupby(["dataset", "arch"])):
        mk = "x" if (ds, a) in EXCLUDED else "o"
        ax.scatter(grp[meas], grp["AS fooling"] * 100, color=c, marker=mk, s=45, label=f"{ds.replace('mnist', '')} {a}")
    ax.set_xscale("log"); ax.set(xlabel=lab)
axes[0].set_ylabel("AS universal fooling (%)"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "phase4_shifted_spectrum.png"), bbox_inches="tight"); plt.show()